# Yol Hasarı Segmentasyonu: SAM ile Çokgen Etiket + YOLO11-seg Eğitimi

İlk notebook'taki tespit modeli hasarın **etrafına kutu** çiziyordu. Bu notebook hasarın **gerçek sınırını
(çokgen)** çizen bir segmentasyon modeli eğitir.

**Sorun:** RDD2022'de sadece kutu etiketi var, çokgen yok.
**Çözüm:** Her kutuyu SAM2'ye (Meta'nın Segment Anything modeli) "bu kutunun içindeki nesneyi bul" diye veriyoruz,
çıkan maskeyi çokgene çeviriyoruz. Buna **sözde etiket** denir: insan değil, model üretmiştir.
Kalitesiz maskelerde (SAM nesneyi bulamadıysa ya da kutuyu tamamen doldurduysa) insanın çizdiği kutu korunur.

**Adil kıyas:** Yeni modeli eski tespit modeliyle, aynı test fotoğraflarında ve **insanların çizdiği orijinal kutulara**
göre karşılaştırıyoruz (bölüm 5). Böylece "segmentasyon kutu doğruluğunu bozdu mu, iyileştirdi mi?" sorusu net cevaplanır.

**Başlamadan önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 GPU` (yoksa T4)

| Adım | Süre (L4, tahmini) |
|---|---|
| Veri indirme + hazırlama | ~10-15 dk |
| SAM dönüşümü (~22 bin görüntü) | ~45-90 dk (ilk 50 görüntüde gösterilen hıza bakın) |
| Eğitim | tespit modelinden ~%30 daha uzun |

## 1. Ortam

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'GPU YOK! Çalışma zamanı → Çalışma zamanı türünü değiştir → L4 veya T4 GPU seçin, sonra baştan çalıştırın.')
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/yol-hasar-tespit'
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

REPO_URL = 'https://github.com/mahmutsamiseyhan/yol-hasar-tespit.git'
!git clone -q {REPO_URL} /content/yol-hasar-tespit || (cd /content/yol-hasar-tespit && git pull -q)
%cd /content/yol-hasar-tespit
!pip install -q ultralytics remotezip

## 2. Veri: RDD2022 indir ve hazırla (ilk notebook'la aynı bölümler)

In [ ]:
!python scripts/download_rdd2022.py --out /content/rdd_raw
!python scripts/prepare_rdd2022.py --src /content/rdd_raw --out /content/rdd_yolo

## 3. SAM ile kutu → çokgen dönüşümü

Etiketler görüntü başına yazılır. Bitince Drive'a yedeklenir; oturum koparsa bu hücre yedekten devam eder.

In [ ]:
import shutil, zipfile
from pathlib import Path
SEG_DIR = Path('/content/rdd_seg')
BACKUP = Path(DRIVE_DIR) / 'rdd_seg_etiketler.zip'
if BACKUP.exists() and not (SEG_DIR / 'labels').exists():
    print('Drive yedeğinden etiketler geri yükleniyor...')
    with zipfile.ZipFile(BACKUP) as z:
        z.extractall(SEG_DIR)

!python scripts/rdd_to_seg_sam.py --src /content/rdd_yolo --out {SEG_DIR} --sam sam2.1_b.pt

with zipfile.ZipFile(BACKUP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in list((SEG_DIR / 'labels').rglob('*.txt')) + [SEG_DIR / '_kaynak.jsonl']:
        z.write(p, p.relative_to(SEG_DIR))
print('Etiket yedeği:', BACKUP)

In [ ]:
# Dönüşüm kalitesi: sınıf bazında SAM'in çokgen üretebildiği oran ve rastgele örnekler
import json
from IPython.display import Image, display
stats = json.load(open(SEG_DIR / 'donusum_istatistik.json'))
names = {'D00': 'Boyuna çatlak', 'D10': 'Enine çatlak', 'D20': 'Timsah sırtı', 'D40': 'Çukur'}
for c, s in stats.items():
    print(f"{c} {names[c]:<14} SAM çokgeni: {s['sam']:>6}  kutu olarak kaldı: {s['kutu']:>6}  (SAM oranı %{100*s['sam_orani']:.0f})")
shutil.copy(SEG_DIR / 'donusum_onizleme.jpg', f'{DRIVE_DIR}/sam_donusum_onizleme.jpg')
display(Image(filename=str(SEG_DIR / 'donusum_onizleme.jpg'), width=1100))

## 4. YOLO11-seg eğitimi

Tespit modeliyle aynı boyut (`s`) ve çözünürlük (640) kullanılır; böylece fark yalnızca segmentasyondan gelir.
Önce `QUICK=True` ile kısa bir deneme yapıp hattın çalıştığını görün, sonra `False` yapın.

In [ ]:
from ultralytics import YOLO

QUICK = True          # İlk denemede True, sonra False
MODEL = 'yolo11s-seg.pt'
EPOCHS = 3 if QUICK else 80
RUN_NAME = 'seg_deneme' if QUICK else 'rdd2022_yolo11s_seg'
PROJECT = f'{DRIVE_DIR}/runs'

last = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'
if last.exists() and not QUICK:
    print('Yarım kalan eğitim bulundu, devam ediliyor:', last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(SEG_DIR / 'data.yaml'), epochs=EPOCHS, imgsz=640, batch=16, patience=20,
        fraction=0.1 if QUICK else 1.0,
        project=PROJECT, name=RUN_NAME, exist_ok=True,
        cos_lr=True, close_mosaic=10, seed=42, plots=True,
    )
RUN_DIR = Path(model.trainer.save_dir)
BEST_SEG = RUN_DIR / 'weights' / 'best.pt'
print('En iyi ağırlıklar:', BEST_SEG)

In [ ]:
for name in ['results.png', 'MaskPR_curve.png', 'BoxPR_curve.png', 'confusion_matrix_normalized.png']:
    p = RUN_DIR / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

## 5. Değerlendirme

**5a. Segmentasyon metrikleri (SAM sözde etiketlerine göre).** Maske mAP'i insan değil SAM çokgenlerine göre
ölçülür; "modelin SAM'i ne kadar taklit edebildiği" anlamına gelir. Gerçek maske doğruluğu, sizin elle
çokgen çizdiğiniz Bursa test setinde ölçülecek (`scripts/evaluate_bursa.py`).

**5b. Adil kutu kıyası (insan kutularına göre).** Eski tespit modeli ile yeni segmentasyon modeli aynı test
fotoğraflarında, orijinal insan kutularına göre ölçülür. Sunumdaki "önce / sonra" rakamı budur.

In [ ]:
import pandas as pd
seg = YOLO(str(BEST_SEG))
ms = seg.val(data=str(SEG_DIR / 'data.yaml'), split='test', imgsz=640, batch=16, plots=True,
             project=PROJECT, name=f'{RUN_NAME}_test', exist_ok=True)
display(pd.DataFrame({
    'kutu (SAM sınırlarından)': [ms.box.mp, ms.box.mr, ms.box.map50, ms.box.map],
    'maske (SAM çokgenlerine göre)': [ms.seg.mp, ms.seg.mr, ms.seg.map50, ms.seg.map],
}, index=['precision', 'recall', 'mAP50', 'mAP50-95']).round(3))

In [ ]:
# 5b. Eski tespit modeli bu klasörde olmalı (ilk notebook'un dışa aktarma hücresi kaydetti)
DET = Path(DRIVE_DIR) / 'best.pt'
assert DET.exists(), f'Tespit modeli bulunamadı: {DET}'
!python scripts/compare_box_map.py --data /content/rdd_yolo/data.yaml --models tespit={DET} segmentasyon={BEST_SEG} --out {DRIVE_DIR}/kiyas_tespit_segmentasyon.json

## 6. Örnek tahminler ve dışa aktarma

In [ ]:
import random, matplotlib.pyplot as plt
test_imgs = sorted(Path('/content/rdd_yolo/images/test').glob('*.jpg'))
sample = random.Random(1).sample(test_imgs, 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, res in zip(axes.flat, seg.predict([str(p) for p in sample], conf=0.35, verbose=False)):
    ax.imshow(res.plot()[:, :, ::-1]); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/ornek_segmentasyon.png', dpi=110); plt.show()

shutil.copy(BEST_SEG, f'{DRIVE_DIR}/best-seg.pt')
print('Drive\'a kaydedildi:', f'{DRIVE_DIR}/best-seg.pt')
print('Bu dosyayı projede models/best-seg.pt olarak koyun.')